In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

os.makedirs(PROCESSED_PATH, exist_ok=True)

# Load raw CSVs
sessions = pd.read_csv(os.path.join(RAW_PATH, "website_sessions.csv"))
orders = pd.read_csv(os.path.join(RAW_PATH, "orders.csv"))
order_items = pd.read_csv(os.path.join(RAW_PATH, "order_items.csv"))
products = pd.read_csv(os.path.join(RAW_PATH, "products.csv"))
refunds = pd.read_csv(os.path.join(RAW_PATH, "order_item_refunds.csv"))
pageviews = pd.read_csv(os.path.join(RAW_PATH, "website_pageviews.csv"))

print("✅ Raw data loaded")

✅ Raw data loaded


In [3]:
def inspect_table(df, name):
    print(f"\n{'='*60}")
    print(f"TABLE: {name}")
    print(f"{'='*60}")
    print("\n--- INFO ---")
    df.info()
    print("\n--- DESCRIBE (numeric columns) ---")
    print(df.describe(include='all'))
    print("\n--- MISSING VALUES ---")
    print(df.isnull().sum())
    print("\n--- FIRST 2 ROWS ---")
    print(df.head(2))

inspect_table(sessions, "sessions")
inspect_table(orders, "orders")
inspect_table(order_items, "order_items")
inspect_table(products, "products")
inspect_table(refunds, "refunds")
inspect_table(pageviews, "pageviews")


TABLE: sessions

--- INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 472871 entries, 0 to 472870
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   website_session_id  472871 non-null  int64 
 1   created_at          472871 non-null  object
 2   user_id             472871 non-null  int64 
 3   is_repeat_session   472871 non-null  int64 
 4   utm_source          389543 non-null  object
 5   utm_campaign        389543 non-null  object
 6   utm_content         389543 non-null  object
 7   device_type         472871 non-null  object
 8   http_referer        432954 non-null  object
dtypes: int64(3), object(6)
memory usage: 32.5+ MB

--- DESCRIBE (numeric columns) ---
        website_session_id           created_at        user_id  \
count        472871.000000               472871  472871.000000   
unique                 NaN               470444            NaN   
top                    NaN  2013-11

In [4]:
# Convert created_at to datetime in all tables
date_tables = {
    'sessions': sessions,
    'orders': orders,
    'order_items': order_items,
    'products': products,
    'refunds': refunds,
    'pageviews': pageviews
}

for name, df in date_tables.items():
    df['created_at'] = pd.to_datetime(df['created_at'])
    print(f"✅ Converted created_at in {name}")

✅ Converted created_at in sessions
✅ Converted created_at in orders
✅ Converted created_at in order_items
✅ Converted created_at in products
✅ Converted created_at in refunds
✅ Converted created_at in pageviews


In [5]:
# Fill missing UTM columns with 'direct' (these are sessions with no marketing source)
utm_cols = ['utm_source', 'utm_campaign', 'utm_content']
for col in utm_cols:
    sessions[col] = sessions[col].fillna('direct')
    print(f"Filled missing {col} with 'direct'")

# Fill missing http_referer with 'none' (direct traffic)
sessions['http_referer'] = sessions['http_referer'].fillna('none')
print("Filled missing http_referer with 'none'")

# Verify no missing values remain
print("\nMissing values after cleaning:")
print(sessions.isnull().sum())

Filled missing utm_source with 'direct'
Filled missing utm_campaign with 'direct'
Filled missing utm_content with 'direct'
Filled missing http_referer with 'none'

Missing values after cleaning:
website_session_id    0
created_at            0
user_id               0
is_repeat_session     0
utm_source            0
utm_campaign          0
utm_content           0
device_type           0
http_referer          0
dtype: int64


In [6]:
# Check for duplicate primary keys
print("Duplicate primary keys:")
print(f"  sessions: {sessions['website_session_id'].duplicated().sum()}")
print(f"  orders: {orders['order_id'].duplicated().sum()}")
print(f"  order_items: {order_items['order_item_id'].duplicated().sum()}")
print(f"  products: {products['product_id'].duplicated().sum()}")
print(f"  refunds: {refunds['order_item_refund_id'].duplicated().sum()}")
print(f"  pageviews: {pageviews['website_pageview_id'].duplicated().sum()}")

# Drop if any (though none expected)
sessions = sessions.drop_duplicates(subset=['website_session_id'])
orders = orders.drop_duplicates(subset=['order_id'])
order_items = order_items.drop_duplicates(subset=['order_item_id'])
products = products.drop_duplicates(subset=['product_id'])
refunds = refunds.drop_duplicates(subset=['order_item_refund_id'])
pageviews = pageviews.drop_duplicates(subset=['website_pageview_id'])

Duplicate primary keys:
  sessions: 0
  orders: 0
  order_items: 0
  products: 0
  refunds: 0
  pageviews: 0


In [7]:
# Extract date components for later analysis
sessions['date'] = sessions['created_at'].dt.date
sessions['hour'] = sessions['created_at'].dt.hour
sessions['day_of_week'] = sessions['created_at'].dt.dayofweek  # Monday=0
sessions['month'] = sessions['created_at'].dt.month
sessions['weekend'] = (sessions['day_of_week'] >= 5).astype(int)

print("Added date features to sessions")

Added date features to sessions


In [9]:
def channel_bucket(source):
    if source == 'direct':
        return 'Direct'
    elif 'gsearch' in str(source):
        return 'Google'
    elif 'bsearch' in str(source):
        return 'Bing'
    elif 'social' in str(source):
        return 'Social'
    else:
        return 'Other'

sessions['channel'] = sessions['utm_source'].apply(channel_bucket)

In [10]:
# Check missing values in sessions (should be 0 after cleaning)
print("Missing values in sessions:")
print(sessions.isnull().sum())

# Check if 'channel' column exists
if 'channel' in sessions.columns:
    print("\n✅ 'channel' column exists. Sample values:")
    print(sessions['channel'].value_counts().head())
else:
    print("\n❌ 'channel' column not found")

Missing values in sessions:
website_session_id    0
created_at            0
user_id               0
is_repeat_session     0
utm_source            0
utm_campaign          0
utm_content           0
device_type           0
http_referer          0
date                  0
hour                  0
day_of_week           0
month                 0
weekend               0
channel               0
dtype: int64

✅ 'channel' column exists. Sample values:
channel
Google    316035
Direct     83328
Bing       62823
Social     10685
Name: count, dtype: int64


In [11]:
# Check data type of created_at in sessions
print(f"created_at dtype: {sessions['created_at'].dtype}")

created_at dtype: datetime64[ns]


In [12]:
import os
PROCESSED_PATH = "../data/processed/"
if os.path.exists(PROCESSED_PATH):
    saved_files = os.listdir(PROCESSED_PATH)
    print(f"Files in {PROCESSED_PATH}:")
    for f in saved_files:
        print(f"  - {f}")
else:
    print("Processed folder not found or empty")

Files in ../data/processed/:
  - .gitkeep
  - fact_sessions.csv
  - orders_cleaned.csv
  - order_items_cleaned.csv
  - pageviews_cleaned.csv
  - products_cleaned.csv
  - refunds_cleaned.csv
  - sessions_cleaned.csv


In [13]:
# Merge
fact_sessions = sessions.merge(
    orders[['website_session_id', 'order_id', 'price_usd']],
    on='website_session_id',
    how='left'
)

# Add conversion flag
fact_sessions['is_converted'] = np.where(fact_sessions['order_id'].notna(), 1, 0)
fact_sessions['price_usd'] = fact_sessions['price_usd'].fillna(0)

print(f"✅ fact_sessions created: {fact_sessions.shape}")
print(f"Conversion rate: {fact_sessions['is_converted'].mean():.2%}")

✅ fact_sessions created: (472871, 18)
Conversion rate: 6.83%


In [14]:
if not refunds.empty and 'order_item_id' in refunds.columns:
    # Aggregate refunds per order_item
    refund_agg = refunds.groupby('order_item_id')['refund_amount_usd'].sum().reset_index()
    
    # Merge to order_items
    order_items_with_refund = order_items.merge(refund_agg, on='order_item_id', how='left')
    order_items_with_refund['refund_amount_usd'] = order_items_with_refund['refund_amount_usd'].fillna(0)
    order_items_with_refund['net_revenue'] = order_items_with_refund['price_usd'] - order_items_with_refund['refund_amount_usd']
    
    # Sum net revenue per order
    net_order_revenue = order_items_with_refund.groupby('order_id')['net_revenue'].sum().reset_index()
    net_order_revenue.rename(columns={'net_revenue': 'net_revenue_usd'}, inplace=True)
    
    # Merge into fact_sessions
    fact_sessions = fact_sessions.merge(net_order_revenue, on='order_id', how='left')
    fact_sessions['net_revenue_usd'] = fact_sessions['net_revenue_usd'].fillna(0)
    print("✅ Net revenue (after refunds) added")
else:
    print("No refunds data or refunds table empty")

✅ Net revenue (after refunds) added


In [15]:
# Save fact_sessions
fact_sessions.to_csv(os.path.join(PROCESSED_PATH, "fact_sessions_clean.csv"), index=False)

# Save individual cleaned tables
sessions.to_csv(os.path.join(PROCESSED_PATH, "sessions_clean.csv"), index=False)
orders.to_csv(os.path.join(PROCESSED_PATH, "orders_clean.csv"), index=False)
order_items.to_csv(os.path.join(PROCESSED_PATH, "order_items_clean.csv"), index=False)
products.to_csv(os.path.join(PROCESSED_PATH, "products_clean.csv"), index=False)
if not refunds.empty:
    refunds.to_csv(os.path.join(PROCESSED_PATH, "refunds_clean.csv"), index=False)
pageviews.to_csv(os.path.join(PROCESSED_PATH, "pageviews_clean.csv"), index=False)

print("✅ All cleaned tables saved to data/processed/")

✅ All cleaned tables saved to data/processed/
